# EXP-VIT-TRANSFER-5SEEDS-007

**NOT EXECUTED.** This independent DeiT-tiny ImageNet transfer experiment uses
the frozen CashewData_Split (8).zip, five fixed seeds, a frozen-backbone head
stage followed by low-rate fine-tuning, and Validation-only checkpoint and
deployment-seed selection. Final Test and the historical real holdout run only
after the five Validation checkpoints are locked. One exact Test duplicate is
excluded from holdout scoring after an explicit byte audit.

The KerasHub preset preprocessor is included exactly once in the saved model:
input is RGB float32 in [0, 255]. Train-only geometric and contrast
augmentation is disabled for Validation, Test and holdout inference.
Training/Validation curves include a measured pre-fit Validation point; no
accuracy is lowered or smoothed for presentation. Confusion matrices use a
white-to-blue scale. Both ZIP exports contain this exact notebook.


In [1]:
# In Colab, run once before imports. Drive API supports shared ZIP fallback.
%pip install -q "keras-hub==0.32.0" scikit-learn pandas matplotlib pillow google-api-python-client


In [2]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import shutil
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import keras
import keras_hub
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

MATRIX_CMAP = LinearSegmentedColormap.from_list(
    "readable_blues", ["#ffffff", "#eaf4ff", "#bfdbfe", "#60a5fa", "#1d4ed8"]
)



def matrix_text_color(value, upper):
    return "#ffffff" if float(value) / max(float(upper), 1.0) >= 0.68 else "#0f172a"


## 1. Frozen configuration

Version 007 is separate from the earlier V05 DeiT 006 and V08 scratch runs.
The V08 archive SHA-256, five class counts and five seeds are fixed before
training. Change any protocol value only in a new experiment ID.


In [3]:
NOTEBOOK_VERSION = "007"
NOTEBOOK_FILENAME = "vit_transfer_5seed_007.ipynb"
EXPERIMENT_ID = "EXP-VIT-TRANSFER-5SEEDS-007"
DATASET_VERSION = "Cashew_dataV08_real_val"
DATASET_ARCHIVE_NAME = "CashewData_Split (8).zip"
DATASET_ARCHIVE_SHA256 = "75C5928B2F6432ACA901BC16831E91F9AA4BDACC1E2FF536753D71DA60C147E7"
DATASET_ARCHIVE_SIZE = 1058695554
DATASET_DRIVE_FILE_ID = "1vaaOwAi72byaxFb7w_ixd_Y2HboKn2-G"
DATASET_ARCHIVE_OVERRIDE = None  # Optional exact mounted ZIP path, e.g. /content/drive/MyDrive/...zip
PRESET = "kaggle://keras/deit/keras/deit_tiny_distilled_patch16_224_imagenet/1"
CLASS_NAMES = ["anthracnose", "healthy", "leaf_miner", "red_rust", "not_cashew_leaf"]
EXPECTED_COUNTS = {
    "train": [939, 803, 875, 1072, 1101],
    "val": [308, 216, 264, 328, 314],
    "test": [122, 118, 132, 158, 156],
}
EXPECTED_TOTALS = {"train": 4790, "val": 1430, "test": 686, "all": 6906}
REQUIRED_ABSENT_IMAGES = [
    "val/leaf_miner/leaf_miner_1056.jpg",
    "val/leaf_miner/leaf_miner_1091.jpg",
    "test/not_cashew_leaf/not_cashew_leaf_1514.JPG",
]
SEEDS = [42, 123, 2026, 3407, 7777]
IMG_SIZE = 224
BATCH_SIZE = 16
HEAD_EPOCHS = 8
FINETUNE_EPOCHS = 12
HEAD_LR = 1e-4
BACKBONE_LR = 5e-6
MIN_LR = 5e-7
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5
RUN_PIPELINE_SMOKE = True
RUN_TRAINING = True
RUN_FINAL_TEST = True
RUN_REAL_HOLDOUT = True  # Historical, previously inspected 48-image set.
REAL_HOLDOUT_ARCHIVE_NAME = "RL_Cashew_holdout.zip"
REAL_HOLDOUT_ARCHIVE_SHA256 = "BDDCB61CA9AF6111E0B26DB7876D0A3309C587AEAFFAEF49B7A9D2D162309165"
REAL_HOLDOUT_ARCHIVE_SIZE = 150455582
REAL_HOLDOUT_DRIVE_FILE_ID = "1PBry_y2bMsvd5_YdmEUy9erSFarS3O52"
REAL_HOLDOUT_EXPECTED_COUNTS = {
    "anthracnose": 13, "healthy": 5, "leaf_miner": 24, "red_rust": 7
}
REAL_HOLDOUT_ARCHIVE_OVERRIDE = None


## 2. Archive and split audit

The notebook pins the V08 ZIP hash, exact class counts, and absent removed
files. Test labels are audited for integrity but never used during fitting or
Validation selection. Historical holdout overlap is checked before training.
If the ZIPs are elsewhere in mounted Drive, the notebook searches by exact
filename. If the shared folder is not mounted, it downloads by the pinned
Drive file IDs from the supplied Dataset folder. Both paths require SHA-256.
An exact path can be set with the two `*_OVERRIDE` settings above.


In [4]:
if Path("/content").is_dir():
    from google.colab import drive

    drive.mount("/content/drive")
    project_root = Path("/content/drive/MyDrive/Cashew_Leaf_model_result")
    archive_path = project_root / "Dataset" / DATASET_ARCHIVE_NAME
    extraction_root = Path("/content/deit_v08_dataset")
    results_root = project_root / "training_results" / "current_dataset"
elif os.name == "nt":
    project_root = Path(r"D:\Study\KhoaLuanTotNghiep\Resource")
    archive_path = Path(r"D:\Study\KhoaLuanTotNghiep\Dataset") / DATASET_ARCHIVE_NAME
    extraction_root = project_root / ".deit_v08_dataset"
    results_root = project_root / "training_results" / "current_dataset"
else:
    raise RuntimeError("Use Google Colab or configure local dataset/results paths.")


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest().upper()


def download_pinned_drive_zip(file_id, destination, expected_name,
                              expected_sha256, expected_size):
    """Fetch a shared Drive ZIP by ID, stream to disk, then verify pinned bytes."""
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    destination = Path(destination)
    if destination.is_file():
        if (destination.stat().st_size == expected_size
                and sha256_file(destination) == expected_sha256):
            return destination
        raise ValueError(f"Existing Drive download has unexpected bytes: {destination}")

    auth.authenticate_user()  # Select the account that can open the shared folder.
    credentials, _ = google.auth.default()
    service = build("drive", "v3", credentials=credentials, cache_discovery=False)
    metadata = service.files().get(
        fileId=file_id, fields="id,name,size,capabilities(canDownload)",
        supportsAllDrives=True).execute()
    if (metadata.get("name") != expected_name
            or int(metadata.get("size", -1)) != expected_size
            or metadata.get("capabilities", {}).get("canDownload") is False):
        raise ValueError(f"Unexpected or inaccessible Drive file metadata: {metadata}")

    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".partial")
    try:
        with partial.open("wb") as stream:
            request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
            downloader = MediaIoBaseDownload(stream, request, chunksize=16 * 1024 * 1024)
            done = False
            while not done:
                _, done = downloader.next_chunk(num_retries=3)
        if partial.stat().st_size != expected_size:
            raise ValueError(f"Incomplete Drive download: {partial.stat().st_size} bytes")
        digest = sha256_file(partial)
        if digest != expected_sha256:
            raise ValueError(f"Drive download SHA-256 mismatch: {digest}")
        partial.replace(destination)
        return destination
    finally:
        partial.unlink(missing_ok=True)


def resolve_dataset_archive(preferred_path, search_roots=None):
    """Locate V08 by exact path or filename, then accept only its pinned bytes."""
    checked = set()
    wrong_hashes = []

    def verify(candidate):
        candidate = Path(candidate).expanduser()
        if not candidate.is_file():
            return None
        identity = str(candidate.resolve())
        if identity in checked:
            return None
        checked.add(identity)
        digest = sha256_file(candidate)
        if digest == DATASET_ARCHIVE_SHA256:
            return candidate, digest
        wrong_hashes.append(f"{candidate} (SHA-256 {digest})")
        return None

    if DATASET_ARCHIVE_OVERRIDE is not None:
        override = Path(DATASET_ARCHIVE_OVERRIDE).expanduser()
        if not override.is_file():
            raise FileNotFoundError(
                f"DATASET_ARCHIVE_OVERRIDE does not exist: {override}. "
                "Use the exact ZIP path shown in the Colab Files panel after Drive mount."
            )
        match = verify(override)
        if match:
            return match
    else:
        match = verify(preferred_path)
        if match:
            return match
        drive_roots = ([Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]
                       if search_roots is None else [Path(root) for root in search_roots])
        for candidate in (
            Path("/content") / DATASET_ARCHIVE_NAME,
            *(root / DATASET_ARCHIVE_NAME for root in drive_roots),
            *(root / "Dataset" / DATASET_ARCHIVE_NAME for root in drive_roots),
        ):
            match = verify(candidate)
            if match:
                return match
        for root in drive_roots:
            if root.is_dir():
                print(f"Searching {root} for {DATASET_ARCHIVE_NAME} ...", flush=True)
                for candidate in root.rglob(DATASET_ARCHIVE_NAME):
                    match = verify(candidate)
                    if match:
                        return match
        if search_roots is None and Path("/content").is_dir():
            print("V08 ZIP is not mounted; downloading the pinned Drive file ID.", flush=True)
            downloaded = download_pinned_drive_zip(
                DATASET_DRIVE_FILE_ID,
                Path("/content/verified_drive_archives") / DATASET_ARCHIVE_NAME,
                DATASET_ARCHIVE_NAME, DATASET_ARCHIVE_SHA256, DATASET_ARCHIVE_SIZE)
            match = verify(downloaded)
            if match:
                return match

    if wrong_hashes:
        raise ValueError(
            "Found ZIP(s) named like V08, but none match the pinned SHA-256 "
            f"{DATASET_ARCHIVE_SHA256}: " + "; ".join(wrong_hashes)
        )
    raise FileNotFoundError(
        f"Cannot find {DATASET_ARCHIVE_NAME}. Mount the Drive account containing "
        "the ZIP, or set DATASET_ARCHIVE_OVERRIDE in the configuration cell "
        "to its exact /content/drive/MyDrive/... path."
    )


archive_path, actual_archive_hash = resolve_dataset_archive(archive_path)
print("Verified V08 dataset ZIP:", archive_path)

if actual_archive_hash != DATASET_ARCHIVE_SHA256:
    raise ValueError(
        f"Wrong V08 archive: {actual_archive_hash}. Expected {DATASET_ARCHIVE_SHA256}."
    )

marker = extraction_root / ".archive_sha256"
if extraction_root.exists():
    if not marker.is_file() or marker.read_text().strip() != DATASET_ARCHIVE_SHA256:
        raise RuntimeError(f"Unverified extraction at {extraction_root}; inspect it before reuse.")
else:
    extraction_root.mkdir(parents=True)
    with zipfile.ZipFile(archive_path) as archive:
        root_resolved = extraction_root.resolve()
        for member in archive.infolist():
            target = (extraction_root / member.filename).resolve()
            if target != root_resolved and root_resolved not in target.parents:
                raise ValueError(f"Unsafe ZIP path: {member.filename}")
        archive.extractall(extraction_root)
    marker.write_text(DATASET_ARCHIVE_SHA256, encoding="utf-8")

dataset_root = extraction_root / "CashewData_Split"
for split in ("train", "val", "test"):
    if not (dataset_root / split).is_dir():
        raise FileNotFoundError(dataset_root / split)

out_dir = results_root / EXPERIMENT_ID
aggregate_dir = out_dir / "aggregate"
package_dir = out_dir / "model_package"
if RUN_TRAINING and out_dir.exists() and any(out_dir.rglob("*.weights.h5")):
    raise FileExistsError(f"Existing checkpoints at {out_dir}; choose a new ID or RUN_TRAINING=False.")
if not RUN_TRAINING and not (out_dir / "experiment_config.json").is_file():
    raise FileNotFoundError("Evaluation-only mode needs a completed saved experiment.")
for folder in (out_dir, aggregate_dir, package_dir):
    folder.mkdir(parents=True, exist_ok=True)

valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
rows = []
for split in ("train", "val", "test"):
    found = sorted(p.name for p in (dataset_root / split).iterdir() if p.is_dir())
    if set(found) != set(CLASS_NAMES):
        raise ValueError(f"Class directories mismatch in {split}: {found}")
    for label, class_name in enumerate(CLASS_NAMES):
        files = sorted(p for p in (dataset_root / split / class_name).rglob("*") if p.is_file())
        if len(files) != EXPECTED_COUNTS[split][label]:
            raise ValueError(f"Unexpected {split}/{class_name} count: {len(files)}")
        for path in files:
            if path.suffix.lower() not in valid_ext:
                raise ValueError(f"Unknown image extension: {path}")
            rows.append({
                "split": split,
                "class": class_name,
                "label": label,
                "relative_path": path.relative_to(dataset_root).as_posix(),
                "sha256": sha256_file(path),
            })

manifest = pd.DataFrame(rows)
manifest.to_csv(out_dir / "dataset_manifest.csv", index=False)
split_overlap = manifest.groupby("sha256")["split"].nunique()
if len(manifest) != EXPECTED_TOTALS["all"]:
    raise ValueError(f"Unexpected total image count: {len(manifest)}")
for relative in REQUIRED_ABSENT_IMAGES:
    if (dataset_root / relative).exists():
        raise ValueError(f"Removed V08 image reappeared: {relative}")
real_counts = {
    split: int(manifest.loc[manifest["split"] == split, "relative_path"]
               .map(lambda p: Path(p).name.startswith("real_")).sum())
    for split in ("train", "val", "test")
}
if real_counts != {"train": 135, "val": 35, "test": 28}:
    raise ValueError(f"Unexpected real_ file counts: {real_counts}")
if (split_overlap > 1).any():
    raise ValueError("Exact duplicate image bytes occur across train/val/test splits.")
manifest_hash = hashlib.sha256(
    manifest[["split", "class", "relative_path", "sha256"]]
    .to_csv(index=False, lineterminator="\n")
    .encode()
).hexdigest()
print("Dataset:", DATASET_VERSION, "images:", len(manifest), "manifest SHA-256:", manifest_hash)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
if not gpus:
    raise RuntimeError("Fine-tuning requires a GPU runtime. Select GPU in Colab.")
print("GPU:", gpus, "TensorFlow:", tf.__version__, "KerasHub:", keras_hub.__version__)

config = {
    "experiment_id": EXPERIMENT_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "external_holdout_protocol": {
        "archive_name": REAL_HOLDOUT_ARCHIVE_NAME,
        "archive_sha256": REAL_HOLDOUT_ARCHIVE_SHA256,
        "drive_file_id": REAL_HOLDOUT_DRIVE_FILE_ID,
        "expected_counts": REAL_HOLDOUT_EXPECTED_COUNTS,
        "gate": "RUN_FINAL_TEST and RUN_REAL_HOLDOUT",
        "separate_from_v08_test": True,
        "macro_scope": "four classes with true holdout support; one exact Test duplicate excluded",
        "selection_source": "validation only",
    },
    "dataset_version": DATASET_VERSION,
    "dataset_archive_sha256": DATASET_ARCHIVE_SHA256,
    "dataset_drive_file_id": DATASET_DRIVE_FILE_ID,
    "dataset_manifest_sha256": manifest_hash,
    "classes": CLASS_NAMES,
    "counts": EXPECTED_COUNTS,
    "seeds": SEEDS,
    "preset": PRESET,
    "training_mode": "ImageNet pretrained, frozen head then low-rate full fine-tune",
    "image_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "head_epochs": HEAD_EPOCHS,
    "finetune_epochs": FINETUNE_EPOCHS,
    "head_lr": HEAD_LR,
    "backbone_lr": BACKBONE_LR,
    "min_lr": MIN_LR,
    "weight_decay": WEIGHT_DECAY,
    "checkpoint_monitor": "val_loss",
    "deployment_seed_metric": "val_macro_f1",
    "test_after_validation_lock": True,
}
config_path = out_dir / "experiment_config.json"
if config_path.exists():
    saved_config = json.loads(config_path.read_text(encoding="utf-8"))
    if saved_config != config:
        raise ValueError("Saved experiment config differs from this run. Use a new experiment ID.")
else:
    config_path.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
environment_path = out_dir / "environment.json"
if not environment_path.exists():
    environment_path.write_text(json.dumps({
        "tensorflow": tf.__version__, "keras": keras.__version__,
        "keras_hub": keras_hub.__version__, "python": platform.python_version(),
        "gpu": [gpu.name for gpu in gpus],
    }, indent=2), encoding="utf-8")



Verified V08 dataset ZIP: /content/verified_drive_archives/CashewData_Split (8).zip
Dataset: Cashew_dataV08_real_val images: 6906 manifest SHA-256: c699a9aca707da25f143db1fc3c9b5133ea2858bcdc468a158e5644e500ec715
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')] TensorFlow: 2.21.0 KerasHub: 0.32.0


## Historical real holdout preflight

The pinned 49-image archive contains one image byte-identical to V08 Test.
The preflight records and excludes precisely that image, then evaluates the
remaining 48. Any additional overlap aborts the experiment. This previously
inspected set is diagnostic evidence, not a new blind field test.


In [5]:
"""Locked external classification evaluation; embedded in standalone notebooks.

This file has no training code. Holdout labels never select checkpoints or seeds.
"""

import hashlib
import json
import os
import stat
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import f1_score, precision_score, recall_score


REAL_HOLDOUT_HELPER_VERSION = "007.1"
REAL_HOLDOUT_CMAP = LinearSegmentedColormap.from_list(
    "real_holdout_readable_blues", ["#ffffff", "#eaf4ff", "#bfdbfe", "#60a5fa", "#1d4ed8"]
)


def holdout_matrix_text_color(value, upper):
    return "#ffffff" if float(value) / max(float(upper), 1.0) >= 0.68 else "#0f172a"

def holdout_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def resolve_holdout_archive(archive_name, override=None):
    if override is not None:
        candidate = Path(override).expanduser()
        if not candidate.is_file():
            raise FileNotFoundError(f"REAL_HOLDOUT_ARCHIVE_OVERRIDE does not exist: {candidate}")
        return candidate  # prepare_real_holdout verifies its pinned SHA-256.

    drive_roots = [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]
    common = [Path("/content") / archive_name]
    for root in drive_roots:
        common.extend([root / "Cashew_Leaf_model_result" / "Dataset" / archive_name,
                       root / "Dataset" / archive_name, root / archive_name])
    if os.name == "nt":
        common.append(Path(r"D:\Study\KhoaLuanTotNghiep\Dataset") / archive_name)
    if Path("/kaggle/input").is_dir():
        common.extend(sorted(Path("/kaggle/input").rglob(archive_name)))

    checked = set()
    wrong_hashes = []
    def verify(candidate):
        if not candidate.is_file():
            return None
        identity = str(candidate.resolve())
        if identity in checked:
            return None
        checked.add(identity)
        digest = holdout_sha256(candidate)
        if digest == REAL_HOLDOUT_ARCHIVE_SHA256:
            return candidate
        wrong_hashes.append(f"{candidate} (SHA-256 {digest})")
        return None

    for candidate in common:
        match = verify(candidate)
        if match:
            return match
    for root in drive_roots:
        if root.is_dir():
            print(f"Searching {root} for {archive_name} ...", flush=True)
            for candidate in root.rglob(archive_name):
                match = verify(candidate)
                if match:
                    return match
    if Path("/content").is_dir():
        print("Holdout ZIP is not mounted; downloading the pinned Drive file ID.", flush=True)
        downloaded = download_pinned_drive_zip(
            REAL_HOLDOUT_DRIVE_FILE_ID,
            Path("/content/verified_drive_archives") / archive_name,
            archive_name, REAL_HOLDOUT_ARCHIVE_SHA256, REAL_HOLDOUT_ARCHIVE_SIZE)
        match = verify(downloaded)
        if match:
            return match
    if wrong_hashes:
        raise ValueError("Holdout ZIP name found but SHA-256 differs from pinned archive: "
                         + "; ".join(wrong_hashes))
    raise FileNotFoundError(
        f"Cannot find {archive_name}. Set REAL_HOLDOUT_ARCHIVE_OVERRIDE in the "
        "configuration cell to its exact mounted ZIP path."
    )


def prepare_real_holdout(archive_path, extraction_dir, class_names, expected_sha256,
                         expected_counts, reference_rows, audit_dir):
    """Verify the frozen archive, labels, decoding and byte overlap with V08.

    reference_rows: dictionaries with path/split and optional existing sha256.
    Exact-byte checking does not establish independence of near-duplicate photos.
    """
    archive_path, extraction_dir, audit_dir = map(Path, (archive_path, extraction_dir, audit_dir))
    actual_hash = holdout_sha256(archive_path)
    if actual_hash != expected_sha256.upper():
        raise ValueError(f"Holdout archive changed: {actual_hash}; expected {expected_sha256}.")
    marker = extraction_dir / ".archive_sha256"
    if extraction_dir.exists():
        if not marker.is_file() or marker.read_text(encoding="utf-8").strip() != actual_hash:
            raise FileExistsError(f"Unverified extraction: {extraction_dir}; choose a new directory.")
    else:
        with zipfile.ZipFile(archive_path) as archive:
            resolved_root = extraction_dir.resolve()
            for member in archive.infolist():
                target = (extraction_dir / member.filename.replace("\\", "/")).resolve()
                is_symlink = stat.S_ISLNK(member.external_attr >> 16)
                if is_symlink or (target != resolved_root and resolved_root not in target.parents):
                    raise ValueError(f"Unsafe ZIP member: {member.filename}")
            extraction_dir.mkdir(parents=True)
            archive.extractall(extraction_dir)
        marker.write_text(actual_hash, encoding="utf-8")

    dataset_root = extraction_dir / "RL_Cashew_holdout"
    if not dataset_root.is_dir():
        raise FileNotFoundError(dataset_root)
    found_classes = sorted(p.name for p in dataset_root.iterdir() if p.is_dir())
    if found_classes != sorted(expected_counts) or not set(found_classes).issubset(class_names):
        raise ValueError(f"Unexpected holdout class folders: {found_classes}")
    rows = []
    for label, name in enumerate(class_names):
        class_dir = dataset_root / name
        files = sorted(p for p in class_dir.rglob("*") if p.is_file()) if class_dir.is_dir() else []
        if len(files) != expected_counts.get(name, 0):
            raise ValueError(f"Unexpected holdout count for {name}: {len(files)}")
        for image_path in files:
            if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
                raise ValueError(f"Unsupported holdout image: {image_path}")
            with Image.open(image_path) as image:
                image.verify()
            with Image.open(image_path) as image:
                image.convert("RGB").load()
            rows.append({"path": str(image_path), "relative_path": image_path.relative_to(dataset_root).as_posix(),
                         "class": name, "label": label, "sha256": holdout_sha256(image_path)})
    manifest = pd.DataFrame(rows)
    if manifest.empty or manifest["sha256"].duplicated().any():
        raise ValueError("Holdout is empty or contains duplicate image bytes.")
    # A marker alone is insufficient: verify cached extraction bytes against the ZIP.
    with zipfile.ZipFile(archive_path) as frozen_archive:
        for row in rows:
            digest = hashlib.sha256()
            with frozen_archive.open("RL_Cashew_holdout/" + row["relative_path"]) as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    digest.update(block)
            if digest.hexdigest().upper() != row["sha256"]:
                raise ValueError(f"Cached holdout extraction changed: {row['relative_path']}")
    audit_dir.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(audit_dir / "dataset_manifest.csv", index=False)

    references = list(reference_rows)
    if not references or not {"train", "val", "test"}.issubset({r["split"] for r in references}):
        raise ValueError("Overlap audit needs the full train/val/test V08 reference manifest.")
    holdout_by_hash = dict(zip(manifest["sha256"], manifest["relative_path"]))
    overlaps = []
    reference_digest = hashlib.sha256()
    for row in references:
        digest = (row.get("sha256") or holdout_sha256(row["path"])).upper()
        identity = row.get("relative_path", str(row["path"]))
        reference_digest.update(f"{row['split']}\\0{identity}\\0{digest}\\n".encode("utf-8"))
        if digest in holdout_by_hash:
            overlaps.append({
                "reference_path": str(row["path"]), "split": row["split"],
                "holdout_relative_path": holdout_by_hash[digest], "sha256": digest
            })
    expected_overlap = {("test", "red_rust/IMG_3244.jpg")}
    observed_overlap = {(row["split"], row["holdout_relative_path"]) for row in overlaps}
    if observed_overlap != expected_overlap or len(overlaps) != 1:
        raise ValueError(f"Unexpected holdout/V08 byte overlap: {overlaps}")
    excluded_paths = {row["holdout_relative_path"] for row in overlaps}
    scored = manifest.loc[~manifest["relative_path"].isin(excluded_paths)].reset_index(drop=True)
    if len(scored) != 48:
        raise ValueError(f"Expected 48 non-overlapping holdout images, found {len(scored)}")
    pd.DataFrame(overlaps).to_csv(audit_dir / "excluded_overlap_images.csv", index=False)
    present = [name for name in class_names if (scored["class"] == name).any()]
    audit = {
        "archive_sha256": actual_hash, "archive_name": archive_path.name,
        "manifest_sha256": hashlib.sha256(
            manifest[["relative_path", "class", "label", "sha256"]].to_csv(index=False).encode()
        ).hexdigest(),
        "source_images": len(manifest), "evaluated_images": len(scored),
        "counts": {name: expected_counts.get(name, 0) for name in class_names},
        "present_classes": present,
        "unsupported_classes": [name for name in class_names if name not in present],
        "reference_images_checked": len(references),
        "reference_fingerprint": reference_digest.hexdigest(),
        "byte_overlap_count": len(overlaps), "byte_overlaps": overlaps,
        "limitation": "Exact bytes only; source-group and near-duplicate independence are not established.",
    }
    (audit_dir / "dataset_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    print("Real holdout:", len(scored), "scored images; one exact Test duplicate excluded")
    return scored, audit


def save_real_holdout_seed(manifest, probabilities, class_names, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.shape != (len(manifest), len(class_names)) or not np.isfinite(probabilities).all():
        raise ValueError("Holdout predictions have invalid shape or non-finite values.")
    if (probabilities < 0).any() or not np.allclose(probabilities.sum(axis=1), 1, atol=1e-5):
        raise ValueError("Holdout evaluator expects probabilities, not logits.")
    y_true = manifest["label"].to_numpy(dtype=np.int32)
    y_pred = probabilities.argmax(axis=1)
    present_ids = sorted(set(y_true.tolist()))
    all_ids = list(range(len(class_names)))
    result = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision_present_classes": float(precision_score(y_true, y_pred, labels=present_ids, average="macro", zero_division=0)),
        "macro_recall_present_classes": float(recall_score(y_true, y_pred, labels=present_ids, average="macro", zero_division=0)),
        "macro_f1_present_classes": float(f1_score(y_true, y_pred, labels=present_ids, average="macro", zero_division=0)),
        "balanced_accuracy_present_classes": float(recall_score(y_true, y_pred, labels=present_ids, average="macro", zero_division=0)),
        "images": len(manifest), "macro_class_count": len(present_ids),
    }
    report = pd.DataFrame(classification_report(y_true, y_pred, labels=all_ids,
        target_names=class_names, output_dict=True, zero_division=0)).transpose()
    per_class = report.loc[class_names, ["precision", "recall", "f1-score", "support"]].copy()
    per_class["evaluated"] = per_class["support"] > 0
    # No evidence for unsupported true classes: don't present placeholder zeros as performance.
    per_class.loc[~per_class["evaluated"], ["precision", "recall", "f1-score"]] = np.nan
    per_class.to_csv(output_dir / "per_class_metrics.csv", index_label="class")
    readable_report = per_class.copy()
    readable_report["scope"] = np.where(readable_report["evaluated"], "true class present", "not evaluated: zero true support")
    readable_report.loc["macro avg (present classes)"] = [
        result["macro_precision_present_classes"], result["macro_recall_present_classes"],
        result["macro_f1_present_classes"], len(manifest), True,
        f"{len(present_ids)} supported true classes only",
    ]
    readable_report.loc["weighted avg"] = [
        report.loc["weighted avg", "precision"], report.loc["weighted avg", "recall"],
        report.loc["weighted avg", "f1-score"], len(manifest), True, "weighted by true support",
    ]
    readable_report.to_csv(output_dir / "classification_report.csv", index_label="class")
    predictions = manifest[["relative_path", "class", "label"]].copy()
    predictions["predicted_label"] = y_pred
    predictions["predicted_class"] = [class_names[i] for i in y_pred]
    predictions["confidence"] = probabilities.max(axis=1)
    predictions["correct"] = y_true == y_pred
    for label, name in enumerate(class_names):
        predictions[f"probability_{name}"] = probabilities[:, label]
    predictions.to_csv(output_dir / "predictions.csv", index=False)
    matrix = confusion_matrix(y_true, y_pred, labels=all_ids)
    for normalized, suffix in ((False, ""), (True, "_normalized")):
        values = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1) if normalized else matrix
        pd.DataFrame(values, index=class_names, columns=class_names).to_csv(output_dir / f"confusion_matrix{suffix}.csv")
        fig, ax = plt.subplots(figsize=(9, 8), facecolor="white")
        ax.set_facecolor("white")
        ax.imshow(values, cmap=REAL_HOLDOUT_CMAP, vmin=0,
                  vmax=1 if normalized else max(int(matrix.max()), 1))
        ax.set_xticks(all_ids, class_names, rotation=40, ha="right", color="black")
        ax.set_yticks(all_ids, class_names, color="black")
        ax.set_xlabel("Predicted class", color="black")
        ax.set_ylabel("True class", color="black")
        ax.set_title("Real holdout" + (" (row normalized)" if normalized else ""), color="black")
        for row in all_ids:
            for col in all_ids:
                text = f"{values[row, col]:.2f}" if normalized else str(int(values[row, col]))
                if normalized and row not in present_ids:
                    text = "N/A"
                ax.text(col, row, text, ha="center", va="center", color=holdout_matrix_text_color(values[row, col], 1 if normalized else matrix.max()))
        fig.tight_layout()
        fig.savefig(output_dir / f"confusion_matrix{suffix}.png", dpi=180)
        plt.close(fig)
    (output_dir / "metrics.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


def evaluate_real_holdout(manifest, audit, class_names, seeds, validation_results,
                          checkpoints, load_model, make_dataset, clear_session,
                          output_dir, outputs_are_logits=False):
    """Freeze all checkpoints and validation selection BEFORE any prediction.

    Completed evaluations are read back, not repeated. Interrupted evaluations
    may resume only with the same checkpoint, validation and archive fingerprints.
    """
    output_dir = Path(output_dir)
    seeds = [int(seed) for seed in seeds]
    validation_results = validation_results.copy()
    if validation_results["seed"].tolist() != seeds:
        raise ValueError("Holdout evaluation requires completed, ordered validation for every seed.")
    if not np.isfinite(validation_results[["val_macro_f1", "val_loss"]].to_numpy()).all():
        raise ValueError("Validation selection metrics are incomplete.")
    ranked = validation_results.sort_values(["val_macro_f1", "val_loss", "seed"], ascending=[False, True, True])
    deployment_seed = int(ranked.iloc[0]["seed"])
    checkpoint_hashes = {str(seed): holdout_sha256(checkpoints[seed]) for seed in seeds}
    lock = {"helper_version": REAL_HOLDOUT_HELPER_VERSION, "archive_sha256": audit["archive_sha256"],
            "manifest_sha256": audit["manifest_sha256"],
            "reference_fingerprint": audit["reference_fingerprint"], "seeds": seeds,
            "checkpoint_sha256": checkpoint_hashes, "deployment_seed": deployment_seed,
            # Canonical float precision avoids false lock changes on CSV reload.
            "validation_sha256": hashlib.sha256(validation_results.to_csv(index=False, float_format="%.12g").encode()).hexdigest(),
            "model_classes": class_names, "macro_classes": audit["present_classes"],
            "selection_source": "validation only; macro F1, val_loss, seed tie-break",
            "outputs_are_logits": outputs_are_logits}
    lock_path = output_dir / "evaluation_lock.json"
    if lock_path.exists():
        if json.loads(lock_path.read_text(encoding="utf-8")) != lock:
            raise ValueError("Holdout lock changed. Do not retune using holdout or overwrite its results.")
    else:
        output_dir.mkdir(parents=True, exist_ok=True)
        lock_path.write_text(json.dumps(lock, indent=2), encoding="utf-8")
    aggregate_dir = output_dir / "aggregate"
    aggregate_dir.mkdir(exist_ok=True)
    results_path = aggregate_dir / "real_holdout_results_5seeds.csv"
    if (output_dir / "evaluation_completed.json").exists():
        print("Real holdout already completed; reading frozen results, not running predictions again.")
        return pd.read_csv(results_path)
    result_rows = []
    for seed in seeds:
        seed_dir = output_dir / f"seed_{seed}"
        completed_path = seed_dir / "evaluation_completed.json"
        if completed_path.exists():
            row = json.loads(completed_path.read_text(encoding="utf-8"))
        else:
            clear_session()
            model = load_model(seed)
            dataset = make_dataset(manifest)
            start = time.perf_counter()
            scores = np.asarray(model.predict(dataset, verbose=0))
            prediction_seconds = time.perf_counter() - start
            if outputs_are_logits:
                exponentials = np.exp(scores - scores.max(axis=1, keepdims=True))
                scores = exponentials / exponentials.sum(axis=1, keepdims=True)
            row = {"seed": seed, **save_real_holdout_seed(manifest, scores, class_names, seed_dir),
                   "prediction_seconds_including_input_pipeline": prediction_seconds}
            completed_path.write_text(json.dumps(row, indent=2), encoding="utf-8")
            del model, scores, dataset
            clear_session()
        result_rows.append(row)
    results = pd.DataFrame(result_rows)
    results.to_csv(results_path, index=False)
    metric_columns = [c for c in results if c not in {"seed", "images", "macro_class_count"}]
    results[metric_columns].agg(["mean", "std"]).transpose().to_csv(aggregate_dir / "real_holdout_mean_std.csv")
    per_class = pd.concat([pd.read_csv(output_dir / f"seed_{seed}" / "per_class_metrics.csv")
                          .assign(seed=seed) for seed in seeds], ignore_index=True)
    per_class.groupby("class")[["precision", "recall", "f1-score"]].agg(["mean", "std"]).to_csv(
        aggregate_dir / "per_class_real_holdout_mean_std.csv")
    deployment_result = next(row for row in result_rows if row["seed"] == deployment_seed)
    (aggregate_dir / "deployment_seed_result.json").write_text(json.dumps({
        "selection_source": "validation only, fixed before holdout", "result": deployment_result,
    }, indent=2), encoding="utf-8")
    (output_dir / "evaluation_completed.json").write_text(json.dumps({
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        "note": "No model or seed selection on holdout. SD measures training-seed variability, not sampling uncertainty.",
    }, indent=2), encoding="utf-8")
    print("Real holdout results (macro metrics over supported true classes only):")
    print(results)
    return results

# --- Notebook-specific real holdout preflight (no prediction/training here) ---
if RUN_FINAL_TEST and RUN_REAL_HOLDOUT:
    holdout_archive = resolve_holdout_archive(REAL_HOLDOUT_ARCHIVE_NAME, REAL_HOLDOUT_ARCHIVE_OVERRIDE)
    holdout_reference_rows = manifest.assign(
        path=manifest["relative_path"].map(lambda relative: str(dataset_root / relative))
    ).to_dict("records")
    holdout_out_dir = out_dir / "real_holdout"
    # Use runtime-local storage, not the Drive result directory, for images.
    if Path("/content").is_dir():
        holdout_extract_dir = Path("/content/real_holdout_007") / REAL_HOLDOUT_ARCHIVE_SHA256[:12]
    elif Path("/kaggle/working").is_dir():
        holdout_extract_dir = Path("/kaggle/working/real_holdout_007") / REAL_HOLDOUT_ARCHIVE_SHA256[:12]
    else:
        import tempfile
        holdout_extract_dir = Path(tempfile.gettempdir()) / "cashew_real_holdout_007" / REAL_HOLDOUT_ARCHIVE_SHA256[:12]
    real_holdout_manifest, real_holdout_audit = prepare_real_holdout(
        holdout_archive, holdout_extract_dir, CLASS_NAMES,
        REAL_HOLDOUT_ARCHIVE_SHA256, REAL_HOLDOUT_EXPECTED_COUNTS,
        holdout_reference_rows, holdout_out_dir,
    )
else:
    print("Real-world holdout remains locked; no holdout extraction/evaluation runs.")


Searching /content/drive/MyDrive for RL_Cashew_holdout.zip ...
Holdout ZIP is not mounted; downloading the pinned Drive file ID.


Real holdout: 48 scored images; one exact Test duplicate excluded


## 3. DeiT-tiny input and model

Images remain RGB float32 [0,255]. The KerasHub preset preprocessor is
called once inside the saved model. Augmentation runs in training mode only.


In [6]:
paths = {}
labels = {}
for split in ("train", "val", "test"):
    part = manifest.loc[manifest["split"] == split]
    paths[split] = [str(dataset_root / p) for p in part["relative_path"]]
    labels[split] = part["label"].to_numpy(dtype=np.int32)


def decode_image(path, label):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    image.set_shape((None, None, 3))
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method="bilinear")
    return tf.cast(image, tf.float32), label


def make_dataset(split, seed, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[split], labels[split]))
    if shuffle:
        ds = ds.shuffle(len(paths[split]), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(seed)


def build_model(seed):
    set_seed(seed)
    for attempt in range(3):
        try:
            classifier = keras_hub.models.DeiTImageClassifier.from_preset(
                PRESET,
                num_classes=len(CLASS_NAMES),
                activation=None,
                head_dtype="float32",
            )
            break
        except ValueError as exc:
            if attempt == 2 or not any(code in str(exc) for code in ("500", "502", "503")):
                raise
            time.sleep(3 * (attempt + 1))
    if classifier.preprocessor is None:
        raise RuntimeError("DeiT preset lacks preprocessing; stop before training.")
    augmentation = keras.Sequential([
        keras.layers.RandomFlip("horizontal", seed=seed + 1),
        keras.layers.RandomRotation(0.04, seed=seed + 2),
        keras.layers.RandomZoom((-0.05, 0.05), (-0.05, 0.05), seed=seed + 3),
        keras.layers.RandomTranslation(0.03, 0.03, seed=seed + 4),
        keras.layers.RandomContrast(0.08, seed=seed + 5),
    ], name="augmentation")
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), dtype="float32", name="image_0_255")
    # Task.preprocessor is normally invoked by Task.fit/predict/evaluate.
    # This classifier is nested in a keras.Model, so call it explicitly.
    processed = classifier.preprocessor(augmentation(inputs))
    outputs = classifier(processed)
    model = keras.Model(inputs, outputs, name="deit_tiny_finetune")
    if model.output_shape != (None, len(CLASS_NAMES)):
        raise RuntimeError(f"Unexpected classifier shape: {model.output_shape}")
    return model, classifier


def compile_model(model, learning_rate):
    model.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=WEIGHT_DECAY),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )


def finetune_lr(epoch, unused_lr):
    if epoch < 3:
        return BACKBONE_LR * (epoch + 1) / 3
    progress = (epoch - 3) / max(1, FINETUNE_EPOCHS - 4)
    return MIN_LR + 0.5 * (BACKBONE_LR - MIN_LR) * (1 + math.cos(math.pi * progress))


def prediction_metrics(y_true, logits):
    y_pred = np.argmax(logits, axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }, y_pred



# Measures image read, decode, resize and tf.data delivery, without model
# inference. This single-pass timing is never used for model selection.
pipeline_started = time.perf_counter()
pipeline_images = 0
for batch_images, _ in make_dataset("val", SEEDS[0], shuffle=False):
    pipeline_images += int(tf.shape(batch_images)[0])
pipeline_seconds = time.perf_counter() - pipeline_started
if pipeline_images != EXPECTED_TOTALS["val"]:
    raise AssertionError("Input pipeline image count changed")
(out_dir / "image_processing_time.json").write_text(json.dumps({
    "images": pipeline_images,
    "seconds": pipeline_seconds,
    "ms_per_image": 1000 * pipeline_seconds / pipeline_images,
    "scope": "single-pass tf.data read, decode, resize and batch delivery; no model inference",
}, indent=2), encoding="utf-8")

if RUN_PIPELINE_SMOKE:
    import tempfile
    set_seed(SEEDS[0])
    smoke_model, smoke_classifier = build_model(SEEDS[0])
    probe = tf.random.stateless_uniform((2, IMG_SIZE, IMG_SIZE, 3),
                                        seed=(SEEDS[0], 7), minval=0, maxval=255)
    direct = np.asarray(smoke_model(probe, training=False))
    manual = np.asarray(smoke_classifier(
        smoke_classifier.preprocessor(probe), training=False))
    if direct.shape != (2, len(CLASS_NAMES)) or not np.isfinite(direct).all():
        raise AssertionError("DeiT pipeline output is invalid")
    if not np.allclose(direct, manual, rtol=1e-5, atol=1e-5):
        raise AssertionError("Nested DeiT preprocessing differs from one explicit pass")
    smoke_classifier.backbone.trainable = False
    compile_model(smoke_model, HEAD_LR)
    if smoke_classifier.backbone.trainable_weights:
        raise AssertionError("Stage 1 backbone is not frozen")
    smoke_classifier.backbone.trainable = True
    compile_model(smoke_model, BACKBONE_LR)
    if not smoke_classifier.backbone.trainable_weights:
        raise AssertionError("Stage 2 backbone is not trainable")
    with tempfile.TemporaryDirectory() as tmp:
        package = Path(tmp) / "pipeline.keras"
        smoke_model.save(package)
        loaded = keras.models.load_model(package, compile=False, safe_mode=True)
        if not np.allclose(direct, np.asarray(loaded(probe, training=False)),
                           rtol=1e-4, atol=1e-4):
            raise AssertionError("Saved DeiT package changed inference")
    (out_dir / "pipeline_smoke.json").write_text(json.dumps({
        "status": "PASS", "preprocessing": "single preset pass",
        "input": "RGB float32 0..255", "stage1_frozen": True,
        "stage2_trainable": True, "saved_model_reload": True,
    }, indent=2), encoding="utf-8")
    del smoke_model, smoke_classifier, loaded
    keras.backend.clear_session()
    gc.collect()


100%|██████████| 615/615 [00:00<00:00, 1.32MB/s]
/usr/local/lib/python3.13/dist-packages/keras/src/saving/serialization_lib.py:791: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)


## 4. Five-seed transfer training

Stage 1 trains the new head with the ImageNet backbone frozen. Stage 2
fine-tunes at a much lower learning rate. Both stage checkpoints are ranked
only by Validation loss. The deployment seed is chosen only from Validation
macro F1, then Validation loss. Training and preprocessing times are saved.


In [7]:
validation_rows = []
training_rows = []
all_histories = []

if RUN_TRAINING:
    for seed in SEEDS:
        print(f"\n===== SEED {seed} =====")
        keras.backend.clear_session()
        gc.collect()
        set_seed(seed)
        seed_dir = out_dir / f"seed_{seed}"
        seed_dir.mkdir(parents=True, exist_ok=True)
        head_path = seed_dir / "head_best.weights.h5"
        ft_path = seed_dir / "finetune_best.weights.h5"
        model_path = seed_dir / "best_model.keras"
        if any(p.exists() for p in (head_path, ft_path, model_path)):
            raise FileExistsError(f"Refusing to overwrite seed {seed} outputs.")

        train_ds = make_dataset("train", seed, shuffle=True)
        train_clean_ds = make_dataset("train", seed, shuffle=False)
        val_ds = make_dataset("val", seed, shuffle=False)
        model, classifier = build_model(seed)
        total_params = int(model.count_params())
        # True pre-fit score: random five-class head, before any update.
        pre_fit_logits = model.predict(val_ds, verbose=0)
        pre_fit_metrics, _ = prediction_metrics(labels["val"], pre_fit_logits)
        (seed_dir / "pre_fit_validation.json").write_text(json.dumps(
            {"seed": seed, **pre_fit_metrics, "images": len(labels["val"])},
            indent=2), encoding="utf-8")
        classifier.backbone.trainable = False
        compile_model(model, HEAD_LR)
        head_start = time.perf_counter()
        h1 = model.fit(
            train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, verbose=1,
            callbacks=[keras.callbacks.ModelCheckpoint(
                str(head_path), monitor="val_loss", mode="min",
                save_best_only=True, save_weights_only=True,
            )],
        )
        head_training_seconds = time.perf_counter() - head_start
        model.load_weights(head_path)
        head_best_loss = float(min(h1.history["val_loss"]))

        classifier.backbone.trainable = True
        compile_model(model, BACKBONE_LR)
        finetune_start = time.perf_counter()
        h2 = model.fit(
            train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS, verbose=1,
            callbacks=[
                keras.callbacks.LearningRateScheduler(finetune_lr, verbose=0),
                keras.callbacks.ModelCheckpoint(
                    str(ft_path), monitor="val_loss", mode="min",
                    save_best_only=True, save_weights_only=True,
                ),
                keras.callbacks.EarlyStopping(
                    monitor="val_loss", mode="min",
                    patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True,
                ),
            ],
        )
        finetune_training_seconds = time.perf_counter() - finetune_start
        ft_best_loss = float(min(h2.history["val_loss"]))
        selected_stage = "finetune" if ft_best_loss < head_best_loss else "head"
        model.load_weights(ft_path if selected_stage == "finetune" else head_path)
        train_clean_loss, train_clean_acc = model.evaluate(train_clean_ds, verbose=0)
        val_loss, _ = model.evaluate(val_ds, verbose=0)
        inference_started = time.perf_counter()
        val_logits = model.predict(val_ds, verbose=0)
        val_inference_ms_per_image = 1000 * (time.perf_counter() - inference_started) / len(labels["val"])
        val_metrics, val_pred = prediction_metrics(labels["val"], val_logits)
        model.save(model_path)
        reloaded = keras.models.load_model(model_path, compile=False, safe_mode=True)
        sample_images = next(iter(val_ds))[0]
        if not np.allclose(np.asarray(model(sample_images, training=False)),
                           np.asarray(reloaded(sample_images, training=False)),
                           rtol=1e-4, atol=1e-4):
            raise AssertionError("Saved model changed Validation inference")
        del reloaded

        history_rows = []
        for stage, history, offset in (("head", h1, 0), ("finetune", h2, HEAD_EPOCHS)):
            for i in range(len(history.history["loss"])):
                history_rows.append({
                    "seed": seed, "stage": stage, "epoch": offset + i + 1,
                    "train_loss": float(history.history["loss"][i]),
                    "train_aug_accuracy": float(history.history["accuracy"][i]),
                    "val_loss": float(history.history["val_loss"][i]),
                    "val_accuracy": float(history.history["val_accuracy"][i]),
                })
        pd.DataFrame(history_rows).to_csv(seed_dir / "history.csv", index=False)
        all_histories.extend(history_rows)

        cm = confusion_matrix(labels["val"], val_pred, labels=range(len(CLASS_NAMES)))
        pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(seed_dir / "validation_confusion_matrix.csv")
        report = classification_report(
            labels["val"], val_pred, target_names=CLASS_NAMES,
            output_dict=True, zero_division=0,
        )
        pd.DataFrame(report).transpose().to_csv(seed_dir / "validation_classification_report.csv")
        pd.DataFrame({
            "relative_path": manifest.loc[manifest["split"] == "val", "relative_path"].to_numpy(),
            "true_label": labels["val"], "predicted_label": val_pred,
        }).to_csv(seed_dir / "validation_predictions.csv", index=False)

        validation_rows.append({"seed": seed, "val_loss": float(val_loss), **{
            f"val_{k}": v for k, v in val_metrics.items()
        }})
        training_rows.append({
            "seed": seed, "selected_stage": selected_stage,
            "head_best_epoch": int(np.argmin(h1.history["val_loss"]) + 1),
            "finetune_best_epoch": int(np.argmin(h2.history["val_loss"]) + 1),
            "head_best_val_loss": head_best_loss,
            "finetune_best_val_loss": ft_best_loss,
            "actual_epochs": len(h1.history["loss"]) + len(h2.history["loss"]),
            "train_clean_loss": float(train_clean_loss),
            "train_clean_accuracy": float(train_clean_acc),
            "head_training_seconds": float(head_training_seconds),
            "finetune_training_seconds": float(finetune_training_seconds),
            "training_seconds": float(head_training_seconds + finetune_training_seconds),
            "val_inference_ms_per_image": float(val_inference_ms_per_image),
            "training_time_scope": "model.fit including epoch validation and callbacks",
            "total_parameters": total_params,
            "checkpoint": str(model_path),
        })
        print("Val:", validation_rows[-1], "Clean train accuracy:", train_clean_acc)
        del model, classifier, train_ds, train_clean_ds, val_ds, h1, h2
        keras.backend.clear_session()
        gc.collect()

    pd.DataFrame(validation_rows).to_csv(aggregate_dir / "validation_results_5seeds.csv", index=False)
    pd.DataFrame(training_rows).to_csv(aggregate_dir / "training_summary_5seeds.csv", index=False)
    pd.DataFrame(all_histories).to_csv(aggregate_dir / "history_5seeds.csv", index=False)
else:
    validation_rows = pd.read_csv(aggregate_dir / "validation_results_5seeds.csv").to_dict("records")
    training_rows = pd.read_csv(aggregate_dir / "training_summary_5seeds.csv").to_dict("records")
    all_histories = pd.read_csv(aggregate_dir / "history_5seeds.csv").to_dict("records")

validation_df = pd.DataFrame(validation_rows)
training_df = pd.DataFrame(training_rows)
if validation_df["seed"].tolist() != SEEDS or training_df["seed"].tolist() != SEEDS:
    raise ValueError("Five-seed results are incomplete or out of order.")
time_columns = [
    c for c in ("head_training_seconds", "finetune_training_seconds", "training_seconds")
    if c in training_df.columns
]
timing_scope = "model.fit including epoch validation and callbacks"
if "head_training_seconds" not in training_df.columns:
    timing_scope = "legacy: fit plus post-fit evaluation and artifact saving"
    print("WARNING: Loaded legacy timings include post-fit work; fit-only time cannot be recovered.")
time_summary = training_df[time_columns].agg(["mean", "std"]).transpose()
time_summary["measurement_scope"] = timing_scope
time_summary.to_csv(aggregate_dir / "training_time_mean_std_seconds.csv")
print("\nTraining time mean and sample SD (seconds):\n", time_summary)
metric_columns = [c for c in validation_df if c.startswith("val_") and c != "val_loss"]
summary = validation_df[metric_columns].agg(["mean", "std"]).transpose()
summary.to_csv(aggregate_dir / "validation_mean_std.csv")
print("\nValidation mean ± sample SD:\n", summary)
per_class_rows = []
summed_val_cm = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=np.int64)
for seed in SEEDS:
    seed_dir = out_dir / f"seed_{seed}"
    report = pd.read_csv(seed_dir / "validation_classification_report.csv", index_col=0)
    for class_name in CLASS_NAMES:
        per_class_rows.append({
            "seed": seed, "class": class_name,
            "precision": float(report.loc[class_name, "precision"]),
            "recall": float(report.loc[class_name, "recall"]),
            "f1": float(report.loc[class_name, "f1-score"]),
        })
    summed_val_cm += pd.read_csv(
        seed_dir / "validation_confusion_matrix.csv", index_col=0
    ).to_numpy(dtype=np.int64)
pd.DataFrame(per_class_rows).groupby("class")[["precision", "recall", "f1"]].agg(
    ["mean", "std"]
).to_csv(aggregate_dir / "per_class_validation_mean_std.csv")
pd.DataFrame(summed_val_cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    aggregate_dir / "validation_confusion_matrix_5seeds_sum.csv"
)

ranked = validation_df.sort_values(["val_macro_f1", "val_loss"], ascending=[False, True])
deployment_seed = int(ranked.iloc[0]["seed"])
source_model = out_dir / f"seed_{deployment_seed}" / "best_model.keras"
deployment_model = package_dir / "deployment_model.keras"
if not deployment_model.exists():
    shutil.copy2(source_model, deployment_model)
elif sha256_file(source_model) != sha256_file(deployment_model):
    raise ValueError("Existing deployment model differs from Validation-selected checkpoint")
(package_dir / "label_map.json").write_text(
    json.dumps({str(i): name for i, name in enumerate(CLASS_NAMES)}, indent=2), encoding="utf-8"
)
(package_dir / "deployment_metadata.json").write_text(json.dumps({
    "experiment_id": EXPERIMENT_ID, "dataset_version": DATASET_VERSION,
    "seed": deployment_seed, "selection": "highest validation macro F1, val_loss tie-break",
    "input": "RGB float32 [0,255], shape [batch,224,224,3]",
    "preprocessing": "included in deployment_model.keras; do not divide by 255 externally",
    "output": "5 logits in CLASS_NAMES order",
    "class_names": CLASS_NAMES,
}, indent=2), encoding="utf-8")
print("Deployment seed selected on validation:", deployment_seed)



# Immutable Validation lock is written before Final Test or holdout inference.
selection_lock = {
    "experiment_id": EXPERIMENT_ID, "dataset_archive_sha256": DATASET_ARCHIVE_SHA256,
    "dataset_manifest_sha256": manifest_hash, "seeds": SEEDS,
    "deployment_seed": deployment_seed,
    "checkpoint_sha256": {
        str(seed): sha256_file(out_dir / f"seed_{seed}" / "best_model.keras")
        for seed in SEEDS
    },
    "validation_sha256": hashlib.sha256(
        validation_df.to_csv(index=False, float_format="%.12g").encode()
    ).hexdigest(),
    "selection_rule": "Validation macro F1, Validation loss tie-break",
}
lock_path = out_dir / "validation_selection_lock.json"
if lock_path.exists():
    if json.loads(lock_path.read_text(encoding="utf-8")) != selection_lock:
        raise ValueError("Validation checkpoint lock changed")
else:
    lock_path.write_text(json.dumps(selection_lock, indent=2), encoding="utf-8")



===== SEED 42 =====
Epoch 1/8


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


300/300 ━━━━━━━━━━━━━━━━━━━━ 29s 55ms/step - accuracy: 0.2850 - loss: 3.0924 - val_accuracy: 0.3441 - val_loss: 2.5021
Epoch 2/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 54ms/step - accuracy: 0.4526 - loss: 1.8299 - val_accuracy: 0.4776 - val_loss: 1.8115
Epoch 3/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 15s 49ms/step - accuracy: 0.5802 - loss: 1.3252 - val_accuracy: 0.5427 - val_loss: 1.4521
Epoch 4/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 53ms/step - accuracy: 0.6683 - loss: 1.0045 - val_accuracy: 0.6210 - val_loss: 1.2104
Epoch 5/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.7205 - loss: 0.8313 - val_accuracy: 0.6657 - val_loss: 1.0576
Epoch 6/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 14s 48ms/step - accuracy: 0.7685 - loss: 0.7055 - val_accuracy: 0.6902 - val_loss: 0.9541
Epoch 7/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 14s 48ms/step - accuracy: 0.7956 - loss: 0.6128 - val_accuracy: 0.7126 - val_loss: 0.8615
Epoch 8/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.8144 - loss: 0.5634 - val_accuracy: 0.73

/usr/local/lib/python3.13/dist-packages/keras/src/saving/serialization_lib.py:791: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)


Val: {'seed': 42, 'val_loss': 0.19180504977703094, 'val_accuracy': 0.9377622377622378, 'val_macro_precision': 0.9366871393527216, 'val_macro_recall': 0.9400268917901355, 'val_macro_f1': 0.9371892258631819, 'val_balanced_accuracy': 0.9400268917901355} Clean train accuracy: 0.9743214845657349

===== SEED 123 =====
Epoch 1/8


...

Val: {'seed': 3407, 'val_loss': 0.1992245614528656, 'val_accuracy': 0.9370629370629371, 'val_macro_precision': 0.9348229550631032, 'val_macro_recall': 0.9394465546593868, 'val_macro_f1': 0.935847067463843, 'val_balanced_accuracy': 0.9394465546593868} Clean train accuracy: 0.9812108278274536

===== SEED 7777 =====
Epoch 1/8


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


300/300 ━━━━━━━━━━━━━━━━━━━━ 26s 55ms/step - accuracy: 0.2386 - loss: 3.5034 - val_accuracy: 0.2210 - val_loss: 3.0189
Epoch 2/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.4000 - loss: 2.0657 - val_accuracy: 0.3881 - val_loss: 2.0875
Epoch 3/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.5363 - loss: 1.4570 - val_accuracy: 0.5091 - val_loss: 1.5584
Epoch 4/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.6386 - loss: 1.0972 - val_accuracy: 0.5930 - val_loss: 1.2812
Epoch 5/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.6971 - loss: 0.9011 - val_accuracy: 0.6503 - val_loss: 1.1033
Epoch 6/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 14s 48ms/step - accuracy: 0.7357 - loss: 0.7721 - val_accuracy: 0.6909 - val_loss: 0.9831
Epoch 7/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.7741 - loss: 0.6625 - val_accuracy: 0.7168 - val_loss: 0.8973
Epoch 8/8
300/300 ━━━━━━━━━━━━━━━━━━━━ 16s 52ms/step - accuracy: 0.7983 - loss: 0.5925 - val_accuracy: 0.73

/usr/local/lib/python3.13/dist-packages/keras/src/saving/serialization_lib.py:791: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)


Val: {'seed': 7777, 'val_loss': 0.1674877256155014, 'val_accuracy': 0.9475524475524476, 'val_macro_precision': 0.9479185051944887, 'val_macro_recall': 0.9486833685994785, 'val_macro_f1': 0.9475324270741667, 'val_balanced_accuracy': 0.9486833685994785} Clean train accuracy: 0.987891435623169

Training time mean and sample SD (seconds):
                                  mean       std  \
head_training_seconds      134.725343  1.511558   
finetune_training_seconds  392.317012  3.989714   
training_seconds           527.042354  4.389974   

                                                           measurement_scope  
head_training_seconds      model.fit including epoch validation and callb...  
finetune_training_seconds  model.fit including epoch validation and callb...  
training_seconds           model.fit including epoch validation and callb...  

Validation mean ± sample SD:
                            mean       std
val_accuracy           0.944056  0.007267
val_macro_precision    0.9

## 5. Validation figures



In [8]:
history_df = pd.DataFrame(all_histories)
fig, axes = plt.subplots(2, len(SEEDS), figsize=(25, 8), sharey="row")
for col, seed in enumerate(SEEDS):
    hist = history_df.loc[history_df["seed"] == seed]
    pre_fit = json.loads((out_dir / f"seed_{seed}" / "pre_fit_validation.json").read_text())
    axes[0, col].plot([0, 1], [pre_fit["accuracy"], hist["val_accuracy"].iloc[0]],
                      color="#ea580c", marker="o", label="full Val from pre-fit")
    axes[0, col].axvline(1, color="#64748b", linestyle=":", alpha=0.4)
    axes[0, col].plot(hist["epoch"], hist["train_aug_accuracy"], label="train augmented")
    axes[0, col].plot(hist["epoch"], hist["val_accuracy"], label="validation")
    axes[1, col].plot(hist["epoch"], hist["train_loss"], label="train")
    axes[1, col].plot(hist["epoch"], hist["val_loss"], label="validation")
    for row in (0, 1):
        axes[row, col].axvline(HEAD_EPOCHS + 0.5, color="gray", linestyle="--", alpha=0.6)
        axes[row, col].set_title(f"Seed {seed}")
        axes[row, col].set_xlabel("Epoch")
        axes[row, col].grid(alpha=0.2)
axes[0, 0].set_ylabel("Accuracy")
axes[1, 0].set_ylabel("Loss")
axes[0, 0].legend()
fig.tight_layout()
fig.savefig(aggregate_dir / "training_curves_5seeds.png", dpi=180)
plt.close(fig)

fig, axes = plt.subplots(1, len(SEEDS), figsize=(26, 5.5))
for col, seed in enumerate(SEEDS):
    cm = pd.read_csv(
        out_dir / f"seed_{seed}" / "validation_confusion_matrix.csv", index_col=0
    ).to_numpy(dtype=np.float64)
    normalized = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    axes[col].imshow(normalized, cmap=MATRIX_CMAP, vmin=0, vmax=1)
    axes[col].set_title(f"Seed {seed}")
    axes[col].set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right", fontsize=8)
    axes[col].set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES, fontsize=8)
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            axes[col].text(j, i, f"{normalized[i, j]:.2f}", ha="center", va="center", fontsize=7, color=matrix_text_color(normalized[i, j], 1))
fig.tight_layout()
fig.savefig(aggregate_dir / "validation_confusion_5seeds.png", dpi=180)
plt.close(fig)



# Per-seed raw and normalized white-blue matrices.
for seed in SEEDS:
    cm = pd.read_csv(out_dir / f"seed_{seed}" / "validation_confusion_matrix.csv",
                     index_col=0).to_numpy()
    for normalized, suffix in ((False, ""), (True, "_normalized")):
        values = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1) if normalized else cm
        fig, ax = plt.subplots(figsize=(8, 7), facecolor="white")
        ax.imshow(values, cmap=MATRIX_CMAP, vmin=0,
                  vmax=1 if normalized else max(1, int(cm.max())))
        ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=40, ha="right")
        ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(f"Validation seed {seed}")
        for row in range(len(CLASS_NAMES)):
            for col in range(len(CLASS_NAMES)):
                ax.text(col, row, f"{values[row,col]:.2f}" if normalized else str(int(values[row,col])),
                        ha="center", va="center",
                        color=matrix_text_color(values[row,col], 1 if normalized else cm.max()))
        fig.tight_layout()
        fig.savefig(out_dir / f"seed_{seed}" / f"validation_confusion_matrix{suffix}.png", dpi=180)
        plt.close(fig)


## 5. Locked Final Test

The five Validation-selected model files and deployment seed are frozen in
validation_selection_lock.json before any Test prediction. Test results cannot
select or change a checkpoint, seed or hyperparameter.


In [9]:
if RUN_FINAL_TEST:
    if json.loads(lock_path.read_text(encoding="utf-8")) != selection_lock:
        raise ValueError("Validation lock changed before Final Test")
    if (aggregate_dir / "test_results_5seeds.csv").exists():
        raise FileExistsError("Final Test already recorded; refusing to overwrite")
    test_rows = []
    per_class_test_rows = []
    test_ds = make_dataset("test", SEEDS[0], shuffle=False)
    for seed in SEEDS:
        model = keras.models.load_model(out_dir / f"seed_{seed}" / "best_model.keras", compile=False)
        inference_started = time.perf_counter()
        logits = model.predict(test_ds, verbose=0)
        test_inference_ms_per_image = 1000 * (time.perf_counter() - inference_started) / len(labels["test"])
        metrics, predictions = prediction_metrics(labels["test"], logits)
        test_rows.append({"seed": seed, "inference_ms_per_image": test_inference_ms_per_image, **{f"test_{k}": v for k, v in metrics.items()}})
        seed_dir = out_dir / f"seed_{seed}"
        cm = confusion_matrix(labels["test"], predictions, labels=range(len(CLASS_NAMES)))
        pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
            seed_dir / "test_confusion_matrix.csv")
        for normalized, suffix in ((False, ""), (True, "_normalized")):
            values = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1) if normalized else cm
            pd.DataFrame(values, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
                seed_dir / f"test_confusion_matrix{suffix}.csv")
            fig, ax = plt.subplots(figsize=(8, 7), facecolor="white")
            ax.imshow(values, cmap=MATRIX_CMAP, vmin=0,
                      vmax=1 if normalized else max(1, int(cm.max())))
            ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=40, ha="right")
            ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
            ax.set_xlabel("Predicted"); ax.set_ylabel("True")
            ax.set_title(f"Final Test seed {seed}")
            for row in range(len(CLASS_NAMES)):
                for col in range(len(CLASS_NAMES)):
                    ax.text(col, row,
                            f"{values[row,col]:.2f}" if normalized else str(int(values[row,col])),
                            ha="center", va="center",
                            color=matrix_text_color(values[row,col], 1 if normalized else cm.max()))
            fig.tight_layout()
            fig.savefig(seed_dir / f"test_confusion_matrix{suffix}.png", dpi=180)
            plt.close(fig)
        report = pd.DataFrame(classification_report(
            labels["test"], predictions, target_names=CLASS_NAMES,
            output_dict=True, zero_division=0,
        )).transpose()
        report.to_csv(seed_dir / "test_classification_report.csv")
        for class_name in CLASS_NAMES:
            per_class_test_rows.append({
                "seed": seed, "class": class_name,
                "precision": float(report.loc[class_name, "precision"]),
                "recall": float(report.loc[class_name, "recall"]),
                "f1": float(report.loc[class_name, "f1-score"]),
            })
        pd.DataFrame({
            "relative_path": manifest.loc[manifest["split"] == "test", "relative_path"].to_numpy(),
            "true_label": labels["test"], "predicted_label": predictions,
        }).to_csv(seed_dir / "test_predictions.csv", index=False)
        del model
        keras.backend.clear_session()
        gc.collect()
    test_df = pd.DataFrame(test_rows)
    test_df.to_csv(aggregate_dir / "test_results_5seeds.csv", index=False)
    test_df.drop(columns="seed").agg(["mean", "std"]).transpose().to_csv(
        aggregate_dir / "test_mean_std.csv"
    )
    pd.DataFrame(per_class_test_rows).groupby("class")[["precision", "recall", "f1"]].agg(
        ["mean", "std"]
    ).to_csv(aggregate_dir / "per_class_test_mean_std.csv")
    print(test_df.drop(columns="seed").agg(["mean", "std"]).transpose())
else:
    print("Final Test remains locked. RUN_FINAL_TEST=False.")


if RUN_FINAL_TEST:
    for normalized, suffix in ((False, ""), (True, "_normalized")):
        fig, axes = plt.subplots(1, len(SEEDS), figsize=(27, 5.8), facecolor="white")
        for column, seed in enumerate(SEEDS):
            values = pd.read_csv(out_dir / f"seed_{seed}" /
                                 f"test_confusion_matrix{suffix}.csv", index_col=0).to_numpy()
            ax = axes[column]
            ax.imshow(values, cmap=MATRIX_CMAP, vmin=0,
                      vmax=1 if normalized else max(1, values.max()))
            ax.set_title(f"Final Test seed {seed}")
            ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right")
            ax.set_yticks(range(len(CLASS_NAMES)))
            ax.set_yticklabels(CLASS_NAMES if column == 0 else [])
            for row in range(len(CLASS_NAMES)):
                for col in range(len(CLASS_NAMES)):
                    ax.text(col, row,
                            f"{values[row,col]:.2f}" if normalized else str(int(values[row,col])),
                            ha="center", va="center", fontsize=7,
                            color=matrix_text_color(values[row,col],
                                                    1 if normalized else values.max()))
        fig.tight_layout()
        fig.savefig(aggregate_dir / f"test_confusion_matrices{suffix}_5seeds.png", dpi=180)
        plt.close(fig)


/usr/local/lib/python3.13/dist-packages/keras/src/saving/serialization_lib.py:791: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)
/usr/local/lib/python3.13/dist-packages/keras/src/saving/serialization_lib.py:791: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)
/usr/local/lib/python3.13/dist-packages/keras/src/saving/serialization_lib.py:791: UserWarning: `compile()` was no

                            mean       std
inference_ms_per_image  7.422890  0.045456
test_accuracy           0.953644  0.004878
test_macro_precision    0.951848  0.005754
test_macro_recall       0.950863  0.005117
test_macro_f1           0.950850  0.005568
test_balanced_accuracy  0.950863  0.005117


## 6. Historical real holdout

Only the 48 images remaining after the exact Test duplicate is excluded are
scored. Four true classes have support; the fifth is N/A. The 5-seed evaluator
uses the immutable Validation-selected checkpoints.


In [10]:
# External evaluation is separate from the internal V08 Test driver above.
def make_real_holdout_dataset(holdout_manifest):
    dataset = tf.data.Dataset.from_tensor_slices((
        holdout_manifest["path"].tolist(),
        holdout_manifest["label"].to_numpy(dtype=np.int32),
    ))
    return dataset.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE).batch(
        BATCH_SIZE, drop_remainder=False
    ).prefetch(tf.data.AUTOTUNE)  # No shuffle or augmentation; same input decoding.


def load_real_holdout_model(seed):
    return keras.models.load_model(holdout_checkpoints[seed], compile=False)


def clear_real_holdout_session():
    tf.keras.backend.clear_session()
    gc.collect()


if RUN_FINAL_TEST and RUN_REAL_HOLDOUT:
    holdout_checkpoints = {seed: out_dir / f"seed_{seed}" / "best_model.keras" for seed in SEEDS}
    real_holdout_results_df = evaluate_real_holdout(
        real_holdout_manifest, real_holdout_audit, CLASS_NAMES, SEEDS, validation_df,
        holdout_checkpoints, load_real_holdout_model, make_real_holdout_dataset,
        clear_real_holdout_session, out_dir / "real_holdout", outputs_are_logits=True,
    )
    display(real_holdout_results_df)
else:
    print("External holdout evaluation skipped. Enable RUN_FINAL_TEST after validation is frozen.")


if RUN_FINAL_TEST and RUN_REAL_HOLDOUT:
    for normalized, suffix in ((False, ""), (True, "_normalized")):
        fig, axes = plt.subplots(1, len(SEEDS), figsize=(27, 5.8), facecolor="white")
        for column, seed in enumerate(SEEDS):
            values = pd.read_csv(out_dir / "real_holdout" / f"seed_{seed}" /
                                 f"confusion_matrix{suffix}.csv", index_col=0).to_numpy()
            ax = axes[column]
            ax.imshow(values, cmap=REAL_HOLDOUT_CMAP, vmin=0,
                      vmax=1 if normalized else max(1, values.max()))
            ax.set_title(f"Historical holdout seed {seed}")
            ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right")
            ax.set_yticks(range(len(CLASS_NAMES)))
            ax.set_yticklabels(CLASS_NAMES if column == 0 else [])
            for row in range(len(CLASS_NAMES)):
                for col in range(len(CLASS_NAMES)):
                    text = f"{values[row,col]:.2f}" if normalized else str(int(values[row,col]))
                    if normalized and not (real_holdout_manifest["label"] == row).any():
                        text = "N/A"
                    ax.text(col, row, text, ha="center", va="center", fontsize=7,
                            color=holdout_matrix_text_color(
                                values[row,col], 1 if normalized else values.max()))
        fig.tight_layout()
        fig.savefig(out_dir / "real_holdout" / "aggregate" /
                    f"holdout_confusion_matrices{suffix}_5seeds.png", dpi=180)
        plt.close(fig)


/usr/local/lib/python3.13/dist-packages/keras/src/saving/serialization_lib.py:791: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)
/usr/local/lib/python3.13/dist-packages/keras/src/saving/serialization_lib.py:791: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)
/usr/local/lib/python3.13/dist-packages/keras/src/saving/serialization_lib.py:791: UserWarning: `compile()` was no

Real holdout results (macro metrics over supported true classes only):
   seed  accuracy  macro_precision_present_classes  \
0    42  0.708333                         0.681818   
1   123  0.708333                         0.688636   
2  2026  0.729167                         0.711851   
3  3407  0.854167                         0.828571   
4  7777  0.729167                         0.687500   

   macro_recall_present_classes  macro_f1_present_classes  \
0                      0.779327                  0.681278   
1                      0.801282                  0.695924   
2                      0.829327                  0.729510   
3                      0.891827                  0.833712   
4                      0.798077                  0.702917   

   balanced_accuracy_present_classes  images  macro_class_count  \
0                           0.779327      48                  4   
1                           0.801282      48                  4   
2                           0.829327

,seed,accuracy,macro_precision_present_classes,macro_recall_present_classes,macro_f1_present_classes,balanced_accuracy_present_classes,images,macro_class_count,prediction_seconds_including_input_pipeline
0,42,0.708333,0.681818,0.779327,0.681278,0.779327,48,4,2.096535
1,123,0.708333,0.688636,0.801282,0.695924,0.801282,48,4,2.172296
2,2026,0.729167,0.711851,0.829327,0.729510,0.829327,48,4,2.199881
3,3407,0.854167,0.828571,0.891827,0.833712,0.891827,48,4,2.112267
4,7777,0.729167,0.687500,0.798077,0.702917,0.798077,48,4,2.168666


## 7. Result ZIPs and exact notebook source

Run after all five seeds and gated evaluations finish.


In [11]:
# Reconstruct the exact output-free notebook even in Colab, where the uploaded
# .ipynb path is not guaranteed to be mounted in the runtime filesystem.
import base64
import zlib
NOTEBOOK_SOURCE_ZLIB_B64 = "..."
source_notebook = json.loads(zlib.decompress(base64.b64decode(NOTEBOOK_SOURCE_ZLIB_B64)))
source_cell = "".join(source_notebook["cells"][-1]["source"])
source_notebook["cells"][-1]["source"] = source_cell.replace(
    'NOTEBOOK_SOURCE_ZLIB_B64 = ""',
    f'NOTEBOOK_SOURCE_ZLIB_B64 = "{NOTEBOOK_SOURCE_ZLIB_B64}"', 1).splitlines(keepends=True)
notebook_bytes = (json.dumps(source_notebook, ensure_ascii=False, indent=1) + "\n").encode("utf-8")
if len(validation_df) != len(SEEDS) or not lock_path.is_file():
    raise RuntimeError("Five Validation seeds and selection lock are required for export")
if RUN_FINAL_TEST and not (aggregate_dir / "test_results_5seeds.csv").is_file():
    raise RuntimeError("Final Test was enabled but has not completed")
if RUN_FINAL_TEST and RUN_REAL_HOLDOUT and not (out_dir / "real_holdout" / "evaluation_completed.json").is_file():
    raise RuntimeError("Real holdout was enabled but has not completed")
for seed in SEEDS:
    if not (out_dir / f"seed_{seed}" / "best_model.keras").is_file():
        raise FileNotFoundError(f"Missing selected model for seed {seed}")
notebook_name = f"{EXPERIMENT_ID}/notebook/{NOTEBOOK_FILENAME}"
zip_paths = {}
for kind in ("FULL", "ANALYSIS"):
    destination = results_root / f"{EXPERIMENT_ID}_{kind}.zip"
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite existing export: {destination}")
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED,
                         compresslevel=3) as archive:
        archive.writestr(notebook_name, notebook_bytes)
        for artifact in sorted(out_dir.rglob("*")):
            if not artifact.is_file():
                continue
            if kind == "ANALYSIS" and artifact.suffix.lower() in {".keras", ".h5", ".weights"}:
                continue
            archive.write(artifact, f"{EXPERIMENT_ID}/{artifact.relative_to(out_dir).as_posix()}")
    with zipfile.ZipFile(destination) as archive:
        if archive.read(notebook_name) != notebook_bytes:
            raise AssertionError("Exported notebook differs from 007 source")
    zip_paths[kind] = str(destination)
    print(kind, destination, "bytes", destination.stat().st_size)
(out_dir / "export_locations.json").write_text(json.dumps(zip_paths, indent=2), encoding="utf-8")


FULL /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-VIT-TRANSFER-5SEEDS-007_FULL.zip bytes 782750036
ANALYSIS /content/drive/MyDrive/Cashew_Leaf_model_result/training_results/current_dataset/EXP-VIT-TRANSFER-5SEEDS-007_ANALYSIS.zip bytes 3147477


272